# Lesson 72 · Fine-tuning Whisper — change just 0.5% of the parameters and teach it a new language: LoRA vs full fine-tuning in practice

**Goal**: fine-tune Whisper with the HuggingFace `Trainer` + LoRA, and use WER to see whether it really learned anything.

> **Engineering lesson**: we state the GPU/time assumptions up front; small datasets overfit easily. Mel parameters are aligned with L47/L70.

🔗 **Aurora connection**: calling `aurora.audio.mel.mel_spectrogram(signal, sr, n_fft=400, hop_length=160, n_mels=80)` lines up with Whisper's preprocessing (25 ms frame length, 10 ms hop, 80 mel channels). Note: the defaults (`n_fft=1024, hop_length=256`) correspond to 64 ms / 16 ms, which is **not** what Whisper uses — you have to pass the parameters explicitly to match. The experiment's findings will go into blog post 2, `docs/current/blog/02_whisper_finetune.md`.

← **Previous lesson** · [L71 · Whisper decoding](L71_whisper_decoding.ipynb)

> Last lesson covered **Whisper decoding strategies**: greedy decoding and beam search, from principle to implementation.
> This lesson explores **fine-tuning Whisper**.

### 🔊 Lining up the audio parameters: how n_fft and hop_length relate to time

Whisper's audio preprocessing uses specific parameters that differ from Aurora's defaults. To reuse Aurora's mel processing, you need to understand what these two parameters mean physically.

**Basic concepts** (recap from L32–L35):
- **Sample rate sr**: how many audio samples are taken per second (unit: Hz). LibriSpeech is 16 kHz, i.e. 16000 samples per second
- **n_fft**: the window size of the fast Fourier transform (in samples). The bigger the window, the finer the frequency resolution and the coarser the time resolution
- **hop_length**: the step between two neighboring windows (in samples). The smaller the step, the finer the time resolution (frames are extracted more densely)

**Converting to time**:
```
frame length = n_fft / sr        (seconds)
frame shift  = hop_length / sr   (seconds)
```

**Whisper's parameters** (official):
- `n_fft = 400`, `sr = 16000` → frame length = 400 / 16000 = **25 ms**
- `hop_length = 160`, `sr = 16000` → frame shift = 160 / 16000 = **10 ms**

**Aurora's defaults** (if you don't specify them):
- `n_fft = 1024`, `sr = 16000` → frame length = 1024 / 16000 = **64 ms**  ❌
- `hop_length = 256`, `sr = 16000` → frame shift = 256 / 16000 = **16 ms**  ❌

**Why line them up?**
Whisper was trained on mel spectrograms with a 25 ms frame length and a 10 ms frame shift. Extract the mel with different parameters and you get features whose distribution differs from the training data, which makes fine-tuning work worse.

**Correct usage** (in this lesson's Aurora calls):
```python
from aurora.audio.mel import mel_spectrogram
mel = mel_spectrogram(signal, sr=16000, n_fft=400, hop_length=160, n_mels=80)
# Note: n_fft=400, hop_length=160 are given explicitly, overriding the defaults 1024/256
```

**Going deeper**:
- 25 ms frame length: long enough to capture the fundamental period of speech (~100 Hz, a 10 ms period) and its harmonics, while keeping enough time resolution
- 10 ms frame shift: matches the time scale on which speech features change (a phoneme usually lasts 50–150 ms, so one phoneme covers 5–15 frames)
- This setting is a rule of thumb from speech processing and has become the standard in Whisper / ASR



> ⚠️ **Cloud environment pre-check**: fine-tuning Whisper-small needs at least **8 GB of GPU memory** (a T4/A10 will get through it; an A100/H100 is faster). Full training on a local CPU isn't realistic (the LibriSpeech-100h subset takes several hours). Recommended platforms: **Colab Pro+ (A100)** or **RunPod (about $1.99/h for an A100) / Lambda Labs**; a single fine-tuning run (1 h on an A100) costs about $2. Detailed setup, resuming from checkpoints and cost control are in `docs/current/course/cloud_gpu_plan.md`.
>
> 💡 **You can learn this locally too**: this notebook treats `transformers / peft / datasets / jiwer / torch` as **optional dependencies that degrade gracefully** — when these packages are missing, the HuggingFace demo cells skip themselves and print a note, while **the pure-numpy exercises (LoRA parameter count, `simple_wer`) and the whiteboard challenge run as usual**, so you can reach this lesson's core learning goals without a GPU.

## Learning goals

By the end of this lesson you should be able to:

1. **Understand the low-rank decomposition behind LoRA**: derive the parameter count for a given rank, and read the output of `print_trainable_parameters`
2. **Inject LoRA with `peft`**: configure `LoraConfig`, and understand what `r`, `lora_alpha` and `target_modules` mean
3. **Configure `Seq2SeqTrainer`**: set the key training arguments, and know how the effective batch is computed
4. **Compute and understand WER**: compute the word error rate with `jiwer`, and explain what S (substitution) / D (deletion) / I (insertion) mean
5. **Adapt Whisper to a low-resource language (Welsh)**: switch the language, swap the dataset, use character-level CER

## This lesson's story: a 244M-parameter model — "change" just 0.5% of it and it speaks Welsh

Whisper-small has 244 million parameters. Fine-tuning all parameters means storing every gradient (244M × float32 ≈ 1 GB), the learning rate is very hard to tune, and a small amount of Welsh data overfits very easily.

The counter-intuitive finding behind **LoRA (Low-Rank Adaptation)**: for most deep-learning tasks the weights only need adjusting inside a **low-rank subspace** — you don't need to change all 244M parameters, only about **0.5% (~1.2M parameters)** of them, and the result comes close to full fine-tuning.

**Analogy**: you want to change the style of a 4K mural. The full approach: repaint every pixel (244M operations). The LoRA approach: just add a thin "style filter" layer (the low-rank matrices A and B) and leave the mural itself frozen.

**The LoRA maths**: for a weight matrix `W ∈ ℝ^{d×k}`:
```
W_new = W + ΔW = W + B·A
where B ∈ ℝ^{d×r}, A ∈ ℝ^{r×k}, r << min(d,k)
parameters: d×r + r×k  vs  the original d×k (full)
```
With `d=k=512, r=8`: LoRA parameters = 512×8 + 8×512 = 8192, original = 262144, **a 97% saving**.

**This lesson's flow**: inject LoRA → HuggingFace Seq2SeqTrainer → WER evaluation → hand-count the parameters as an exercise.

## 🖼️ rank=2 in one picture: why "two skinny matrices" can change a big one

Freeze the original weight `W` and train only **a low-rank branch added alongside it**, `B·A`. The shapes alone show where the savings come from:

```
         full fine-tuning                      LoRA (rank r=2)
   ┌─────────────────┐          ┌─────────────────┐   ┌──┐
   │                 │          │                 │   │  │
   │        W        │          │        W        │ + │B │ ·  ┌──────────┐
   │  d × k (frozen) │          │  d × k (frozen) │   │  │    │    A     │
   │                 │          │                 │   │  │    │  r × k   │
   └─────────────────┘          └─────────────────┘   └──┘    └──────────┘
    updates all d×k params         W stays put; train only B(d×r) and A(r×k)
                                   params = d·r + r·k  ≪  d·k
```

**Intuition**: `B·A` is the product of two long, thin matrices. The result is still a `d×k` "patch ΔW", but it is confined to a low-dimensional subspace of **rank ≤ r**. It's like laying a thin filter over a big painting — using very few degrees of freedom to "touch up" `W` rather than "repaint" all of it.

Take `d=k=768, r=8`: full `768² ≈ 590K`, LoRA `2×768×8 ≈ 12K` — **for the same weight, ~98% fewer parameters**. (The exercise below has you work this out yourself.)


### 🔍 Order of multiplication: why B·A and not A·B?

This needs spelling out, because the LoRA literature uses both notations and they're easy to mix up.

**The standard definition** (used in this lesson):
```
ΔW = B @ A
where B ∈ R^{d × r},  A ∈ R^{r × k}
result: ΔW ∈ R^{d × k}  (same shape as the original W)
```

**Why this order?**
- Matrix multiplication needs the inner dimensions to match: `(d × r)` times `(r × k)` → both inner dimensions are `r` ✓
- The result has shape `(d × k)` ✓
- The other way round, `A @ B` would be `(r × k)` times `(d × r)` → inner dimensions `k ≠ d`, **can't be multiplied** ✗

**The clever bit — the initialization strategy**:
- `B = 0` initialization (zero matrix)
- `A ~ N(0, σ²)` random initialization
- Forward pass: `ΔW = B @ A = 0 @ A = 0` (an all-zero matrix)
- **Result**: at the start of training, `W_new = W + 0 = W`, exactly equivalent to the original model — the model's initial behavior doesn't change

This neat initialization guarantees that fine-tuning doesn't start from a "scrambled" state.

**How the parameters get updated**:
1. Forward: input `x` → compute `y = (W + B @ A) @ x`
2. Backward: compute gradients only for B and A; no gradient is computed for W (frozen)
3. Optimizer: update only A and B; W stays unchanged


### 📐 An intuitive picture of the low-rank subspace

"Low-rank subspace" sounds abstract, but it's really just a simple constraint.

**An everyday analogy**: imagine you want to change a 1000×1000-pixel mural (d=k=1000).
- **Full fine-tuning**: you can change each pixel's RGB values independently (a million degrees of freedom) → too many parameters, high risk of overfitting
- **The LoRA way**: instead of changing pixels directly, you put a "projector" over the mural; the projector uses 8×1000 parameters to control the horizontal projection and 1000×8 parameters for the vertical projection (16000 degrees of freedom in total)
  - However you project, the final effect is decided by combinations of these two "directions" — it can't produce a completely arbitrary change
  - That is what "confined to a low-dimensional subspace of rank ≤ 8" means

**In mathematical language**:
- The rank of `ΔW = B @ A` is at most `min(d_of_B, k_of_A) = min(d, r) = r`
- whereas a fully independent `d×k` matrix has rank at most `min(d, k)`
- so the weight change expressed by `B·A` is limited to the subspace of rank at most `r`

**Why does this work?**
There's an empirical finding in deep learning: for most tasks, the weight adjustments actually happen in a low-dimensional subspace. Even with a million parameters, the directions of change that really "matter" may number only a few dozen. LoRA builds on this observation, forcing the model's weight updates to move only within a space of rank ≤ r, which cuts parameters and guards against overfitting.



### 🎯 Why inject LoRA only into q_proj and v_proj?

Transformer self-attention has four key linear projection matrices: q_proj, k_proj, v_proj and out_proj. Why does LoRA change only q_proj and v_proj?

**What each one does**:
```
input x
  ↓
[q_proj] → query Q      ← asks "what to look for"
[k_proj] → key K        ← decides "what gets looked at"
[v_proj] → value V      ← decides "what value to take"
  ↓
attention weights = softmax(Q·K^T / √d)
  ↓
output = attention weights · V
  ↓
[out_proj] → final output
```

**Why are Q and V more important?**

1. **Q (query) decides what the model is asking for**:
   - Q encodes "what do I want to look for in this sentence right now"
   - The most direct effect on task fine-tuning: it changes "the angle of looking"
   - Example: Whisper's training data is mostly English, so Q is tuned to "the angle for recognizing English sounds"; when you fine-tune for Welsh, Q needs to shift to "the angle for recognizing Welsh sounds"

2. **V (value) decides what gets output**:
   - V encodes "which features to take once attention has focused"
   - It directly affects the final feature representation
   - Change V and you change the features that get extracted — the most direct lever for the downstream task

3. **Why doesn't it matter much whether K and Out are changed?**
   - K only affects attention's "similarity computation", and usually contributes less to task adaptation than Q/V
   - Out is just a linear combination of the heads' outputs, and can be adjusted indirectly through the changes to Q/V
   - Experiments show that changing only Q+V already reaches 80%+ of full fine-tuning's performance

4. **Parameter-efficiency comparison**:
   ```
   only q_proj + v_proj (2 matrices): parameters ≈ 2 × 2 × d × r = 4dr
   q+k+v+out (4 matrices):            parameters ≈ 4 × 2 × d × r = 8dr
   ```
   Choosing q+v gets almost the same result with 50% of the parameters

**Rule of thumb**: when resources are limited, change q_proj and v_proj first; if that still falls short, extend to other layers.



### 📊 Why two matrices B·A rather than one low-rank matrix C?

This is a natural question: instead of two matrices, why not just learn one low-rank matrix C (rank ≤ r) and be done?

**Answer**: the two-matrix design has several deeper advantages.

**1. The clever initialization (the most important)**

With two matrices:
- `B = 0` (zero initialization)
- `A ~ N(0, σ²)`
- Result: `ΔW = B @ A = 0` (the starting point is the original model; no warm-up needed)

With one matrix:
- `C` has to be initialized to some low-rank matrix (it can't be all zeros, or it can't learn)
- Even if it starts very small (`C ~ 0.01×N(0,1)`), it immediately perturbs the weights
- The pretrained model's performance gets damaged, and it takes more epochs to recover

**Side-by-side in practice**:
```
B·A init: W_new = W + 0 = W  (original model fully preserved) → useful fine-tuning starts right away
C init:   W_new = W + C      (original model slightly perturbed) → the first few epochs are spent "repairing" the initial perturbation
```

**2. Stable gradient flow**

With the factorization B @ A, backpropagation passes through two matrices:
```
∂L/∂A = B^T @ (∂L/∂ΔW)   # the gradient A receives is scaled by the size of B
∂L/∂B = (∂L/∂ΔW) @ A^T   # the gradient B receives is scaled by the size of A
```
These two gradients are naturally scaled by different factors, which avoids the gradients becoming too large or too small, as can happen with a single matrix.

With a single matrix C:
```
∂L/∂C = ∂L/∂ΔW   # the gradient is passed straight through, with no scaling
```
This makes exploding or vanishing gradients more likely, especially when the rank is small.

**3. Numerical stability**

The product B @ A naturally keeps the matrix norm in a reasonable range:
- the update is confined to a rank-≤ r subspace and B and A are each small, so the size of ΔW stays fairly controlled (and at the start B=0, so ΔW is exactly 0)

A single matrix C needs careful initialization and regularization to behave similarly.

**The evidence**:
The original LoRA paper (Hu et al., 2021) uses exactly this B=0 + A~N(0,σ²) scheme so that ΔW = 0 when training starts and the model begins as the pretrained one. It's not just a small engineering trick, but a design choice with a real mathematical reason behind it.



In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
# Needs a cloud GPU; dependencies: transformers, peft, datasets, jiwer
# pip install transformers peft datasets jiwer
# These are optional: without them the pure-numpy exercises and the whiteboard still run (see "Cloud environment pre-check" at the top).
try:
    from transformers import WhisperForConditionalGeneration, WhisperProcessor
    from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
    from peft import LoraConfig, get_peft_model
    HAS_HF = True
except ImportError:
    HAS_HF = False
    print("⏭️ transformers/peft not installed (local CPU environment).")
    print("   The HuggingFace demo cells below will skip themselves; the pure-numpy exercises and the whiteboard run as usual.")
    print("   Cloud setup: see docs/current/course/cloud_gpu_plan.md")

## 1. LoRA (Low-Rank Adaptation)

For a weight matrix `W ∈ R^{d × k}`, LoRA freezes `W` and introduces an extra:

```
ΔW = B @ A       B ∈ R^{d × r},  A ∈ R^{r × k},  r << min(d, k)
```

The forward pass becomes `y = (W + ΔW) x = W x + B @ A @ x`.

**Initialization strategy (crucial)**:
- `B = 0` (zero-matrix initialization)
- `A ~ N(0, σ²)` (Gaussian random initialization, usually σ=1/√r)

The beauty of this: **at the start of training, `ΔW = 0 @ A = 0`, so `W_new = W + 0 = W`, exactly equivalent to the original model**. That guarantees fine-tuning starts from a good starting point — the original model — rather than from a random, "scrambled" perturbation. As training goes on, B gradually grows from the zero matrix into a meaningful one, and only then does ΔW start to affect the original weights.

For Whisper-small (`d_model = 768`), `q_proj` and `v_proj` in self-attention are each `768 × 768`. Take `r = 8`: the original trainable parameters are about `2 × 768² ≈ 1.18 M`; the LoRA parameters are `2 × 2 × 768 × 8 = 24 576`, **a 98% reduction**.

In [ ]:
# Demo: load the model, inject LoRA, print the share of trainable parameters
if HAS_HF:
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,          # scaling factor: ΔW is multiplied by lora_alpha / r = 4
        # lora_alpha in detail:
        #   In practice, ΔW = B @ A is added to the original weight W as a correction.
        #   But if ΔW were used as is, then when r is small, even if A/B train to large values,
        #   the absolute size of ΔW could still be too small, so its effect on W is weak.
        #   To compensate, the peft library applies a scaling in the forward pass:
        #     W_effective = W + (lora_alpha / r) * ΔW
        #   This way, whatever r is (4/8/16), the actual scaling factor stays reasonable.
        #   Example: r=8, lora_alpha=32 → scaled 4×; r=16, lora_alpha=32 → scaled 2×.
        #   This keeps the training hyperparameters fairly stable: change r without having to retune the learning rate much.
        #
        #   Physical meaning: think of lora_alpha as the "gain" on the ΔW "patch";
        #   it doesn't change the learning rate (that would affect all weights) but amplifies the LoRA branch alone.
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_2_SEQ_LM",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    # print_trainable_parameters() injects A/B low-rank branches into every matching q_proj/v_proj module
    # (encoder self-attention + decoder self-attention + decoder cross-attention, dozens of matrices in all);
    # the share of trainable parameters is usually far below 1% (trainable% < 1).
else:
    print("⏭️ transformers/peft not installed (local CPU environment).")
    print("   The HuggingFace demo cells below will skip themselves; the pure-numpy exercises and the whiteboard run as usual.")
    print("   Cloud setup: see docs/current/course/cloud_gpu_plan.md")

## 2. HuggingFace `Seq2SeqTrainer`

`Seq2SeqTrainer` adds **generative evaluation** (beam search) on top of `Trainer`, suited to encoder-decoder models. The key `TrainingArguments`:

| Argument | Meaning |
|---|---|
| `per_device_train_batch_size` | batch per GPU, affects memory; 16 is common |
| `gradient_accumulation_steps` | accumulate N steps before updating, effectively enlarging the batch |
| `fp16` | mixed precision; can be turned on for A100/V100 |
| `predict_with_generate` | call `model.generate()` at eval time to compute the real WER |
| `generation_max_length` | maximum number of tokens to decode (225 for LibriSpeech sentences) |
| `save_steps` / `eval_steps` | save a checkpoint / run eval every N steps |

### Gradient accumulation in detail

**The situation**: your GPU memory only fits batch_size=16, but for stable training with less noise you want an effective batch of 32.

**The solution**: gradient accumulation
```
gradient_accumulation_steps = 2   # accumulate 2 steps
per_device_train_batch_size = 16
```

**Step by step**:
1. **Step 1** (step 0): load 16 samples → forward → backward to compute gradients → **don't update the parameters**; the gradients are kept in GPU memory
2. **Step 2** (step 1): load another 16 samples → forward → backward, **the gradients are added to the previous ones** → **only now does the optimizer update the parameters**
3. **Effect**: equivalent to updating with the combined gradient of 32 samples, while the memory only ever holds 16 samples at a time

**Mathematical equivalence**:
```
single update, batch=32:      g_merged = ∇L(all 32 samples)  →  θ ← θ - α·g_merged
accumulated, batch=16×2:      g1 = ∇L(samples 0-15)
                              g2 = ∇L(samples 16-31)
                              g_merged = g1 + g2  →  θ ← θ - α·g_merged
```
The gradients are exactly the same either way.

### The effective-batch formula

```
effective batch_size = per_device_train_batch_size × gradient_accumulation_steps × num_gpus
```

**Why multiplication and not addition?**
- `per_device_train_batch_size=16`: each GPU loads 16 samples at a time
- `gradient_accumulation_steps=2`: accumulate twice, doubling the samples → 16×2 = 32
- `num_gpus=4`: 4 GPUs work in parallel, **each GPU accumulates 32 samples, and after the gradients are synchronized they update together** → 32×4 = 128

Wait — this needs clarifying: **in a PyTorch DDP (Distributed Data Parallel) setup**, the optimizer update happens after gradient synchronization, so the effective batch per GPU is `per_device × accumulation_steps`, and the globally synchronized gradient combines all the GPUs. So the formula holds.

**A practical example** (single A100 40GB):
- `batch_size=16, accumulation=2`: actual batch ≈ 32 (memory ~25 GB)
- `batch_size=32, accumulation=1`: actual batch = 32 (memory ~38 GB)
- These two settings converge the same way, but the first leaves more memory headroom for other uses (such as more gradient checkpointing, lower precision, etc.).

In [ ]:
# Demo: load the dataset + Processor, build the DataCollator, configure TrainingArguments
if HAS_HF:
    from datasets import load_dataset
    from transformers import DataCollatorForSeq2Seq
    import torch

    processor = WhisperProcessor.from_pretrained(
        "openai/whisper-small", language="English", task="transcribe"
    )

    # Download only train-clean-100 (about 6 GB) and test-clean for a quick demo
    # For full training, drop the split restriction
    dataset = load_dataset(
        "librispeech_asr", "clean",
        split={"train": "train.clean.100", "test": "test.clean"},
        trust_remote_code=True,
    )

    def preprocess(batch):
        audio = batch["audio"]
        batch["input_features"] = processor(
            audio["array"], sampling_rate=audio["sampling_rate"],
            return_tensors="pt"
        ).input_features[0]
        batch["labels"] = processor.tokenizer(batch["text"]).input_ids
        return batch

    dataset = dataset.map(preprocess, remove_columns=dataset["train"].column_names, num_proc=4)

    data_collator = DataCollatorForSeq2Seq(processor.tokenizer, model=model, padding=True)

    training_args = Seq2SeqTrainingArguments(
        output_dir="./whisper-small-lora-librispeech",
        per_device_train_batch_size=16,
        gradient_accumulation_steps=2,    # effective batch = 32
        learning_rate=1e-4,
        num_train_epochs=3,
        fp16=torch.cuda.is_available(),
        predict_with_generate=True,
        generation_max_length=225,
        save_steps=500,
        eval_steps=500,
        eval_strategy="steps",      # new name in transformers>=4.38 (the old name evaluation_strategy is deprecated)
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,          # lower WER is better
        report_to="tensorboard",
    )
    print("TrainingArguments configured")
else:
    print("⏭️ Skipping dataset loading and TrainingArguments setup (needs datasets/transformers, and downloads ~6 GB of LibriSpeech).")
    print("   effective batch = per_device_train_batch_size × gradient_accumulation_steps × num_gpus.")

## 3. Evaluation metric: word error rate (WER)

> **How L72 and L73 fit together**: this section covers the background of WER and how to compute it; L73 builds on it to implement the edit-distance back-trace, count S / D / I rates sentence by sentence and pinpoint high-WER sentences, without repeating the basic definitions.

```
WER = (S + D + I) / N
```

- `S`: substitution; `D`: deletion; `I`: insertion
- `N`: total number of words in the reference text

The `jiwer` library computes the edit distance with dynamic programming in milliseconds. LibriSpeech test-clean is read English speech; the baseline (zero-shot Whisper-small) is about **3.4% WER** (as reported in the Whisper paper), and after fine-tuning on the 100h subset the target is **to beat the baseline**, possibly getting close to 2–3%.

Note: WER is sensitive to case and punctuation, so **normalize** before computing it (`WhisperProcessor` has a built-in `normalizer`).

### What does WER > 1.0 mean?

**WER can be greater than 100%**. That looks odd, but it makes sense: WER measures "how wrong the model is", not "the probability of an error".

**A real example**:
```
reference (ref):   "good"                              (1 word)
model output (hyp): "this is very very good"           (5 words)
```
Minimum number of edit operations: **remove the model's 4 extra words — these count as insertions (I=4)** — to get back to ref
```
WER = (S + D + I) / N = (0 + 0 + 4) / 1 = 4.0 = 400%
```

**What problem does it point to in practice?**
- **WER > 1.0**: the model's output is much longer than the reference text — it produced a lot of hallucinated words
- **Possible causes**:
  - labeling errors in the data (the reference text itself is incomplete)
  - a decoding-strategy problem (e.g. a badly set length penalty in beam search)
  - extreme overfitting or a learning rate set too high
  - corrupted or silent audio, which makes the model "babble"
- **What to do about it**:
  - check the dataset's quality (spot-check reference texts at random)
  - adjust the decoding parameters (lower length_penalty, reduce num_beams)
  - lower the learning rate or add regularization

On this lesson's LibriSpeech, with high-quality data and sensible fine-tuning, WER > 1 isn't expected.

In [ ]:
# Demo: compute WER with jiwer + set up the compute_metrics callback
try:
    import jiwer
    HAS_JIWER = True
except ImportError:
    HAS_JIWER = False

# compute_metrics needs jiwer + the HuggingFace processor/normalizer
if HAS_JIWER and HAS_HF:
    from transformers.models.whisper.english_normalizer import BasicTextNormalizer

    normalizer = BasicTextNormalizer()

    def compute_metrics(pred):
        pred_ids = pred.predictions
        label_ids = pred.label_ids

        # [KEY] Why replace -100?
        # In Seq2Seq tasks the samples in a batch have different lengths and must be padded to the same length.
        # The padding tokens should be "ignored" and take no part in the loss.
        # HuggingFace's convention: the special marker -100 means "this is padding, ignore it"
        # (-100 is a "dummy value" that matches no real token in the vocabulary).
        # At evaluation time we have to turn -100 back into the real pad token ID so tokenizer.decode() can handle it.
        # Otherwise you get strange symbols or garbage.
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

        pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
        label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
        # Normalization: lowercase, strip punctuation, etc.
        pred_str  = [normalizer(s) for s in pred_str]
        label_str = [normalizer(s) for s in label_str]
        wer = jiwer.wer(label_str, pred_str)
        return {"wer": wer}

# Quick unit test: is the WER computed correctly? (needs only jiwer)
if HAS_JIWER:
    ref  = ["the cat sat on the mat"]
    hyp  = ["the cat sat on a mat"]          # 1 substitution, 6 words in total
    wer_val = jiwer.wer(ref, hyp)
    assert abs(wer_val - 1/6) < 1e-6, f"WER computed wrongly: {wer_val}"
    print(f"✅ WER unit test passed: WER = {wer_val:.4f} (expected {1/6:.4f})")
else:
    print("⏭️ jiwer not installed, skipping the WER demo; the pure-numpy simple_wer exercise below is unaffected.")

## ✏️ Exercises: count LoRA parameters by hand + implement a simplified WER

### Exercise 1: counting parameters

Given the following setup (matching the variables in the code cell below):
- `d_model = 512`, `r = 16` (LoRA rank)
- one attention layer has three `512×512` matrices, `q_proj`, `k_proj`, `v_proj` (`num_matrices = 3`)

**Working it out by hand**:
| Step | Formula | Result |
|---|---|---|
| original parameters per matrix | `d×d` = `512×512` | 262144 |
| LoRA parameters per matrix | `d×r + r×d` = `2×512×16` | 16384 |
| LoRA total for 3 matrices | `3 × 16384` | 49152 |
| full-parameter baseline for 3 matrices | `3 × 512²` | 786432 |
| reduction | `(1 − 49152/786432) × 100%` | **≈ 93.75%** |

> Switch to `r = 8`, or inject into only the two q/v matrices, and the reduction gets even bigger — exactly why this lesson's demo configuration picks only `q_proj`/`v_proj`.

### Exercise 2: implement a simplified WER

**Signature**:
```python
def simple_wer(ref: str, hyp: str) -> float:
    # ref: reference text (ground truth)
    # hyp: model output text (hypothesis)
    # returns: WER ∈ [0.0, ∞), WER=(S+D+I)/N
```

**How to implement it**: use edit distance. Split `ref` and `hyp` on spaces, compute the word-level edit distance (the same as `edit_distance()` in L67/L68), and divide by the number of words in ref.

**Acceptance criteria**:
- `simple_wer("hello world", "hello world")` == 0.0
- `simple_wer("hello world", "hello")` == 0.5 (1 word deleted, 2 words in total)
- `simple_wer("a b c", "x b c")` ≈ 0.333 (1 word substituted)

### Edit distance at a glance (groundwork for Exercise 2)

Before implementing `simple_wer`, a quick recap of the dynamic-programming method for edit distance (Levenshtein distance). It's a classic algorithm that computes the minimum number of operations (insertions, deletions, substitutions) needed to turn one sequence into another.

**Problem**: given two word lists
```python
ref_words = ["hello", "world"]        # reference text, length m=2
hyp_words = ["hello"]                 # model output, length n=1
```
find the minimum number of operations.

**The DP table** (an `(m+1) × (n+1)` matrix):
- row index: the first i words of ref (i=0 to m)
- column index: the first j words of hyp (j=0 to n)
- `dp[i][j]`: the minimum number of operations to turn the first i words of ref into the first j words of hyp

**Initialization**:
- `dp[0][j] = j` (insert j words starting from the empty sequence)
- `dp[i][0] = i` (delete i words down to the empty sequence)

**Recurrence**:
```
if ref_words[i-1] == hyp_words[j-1]:
    dp[i][j] = dp[i-1][j-1]                    # same word, no operation needed
else:
    dp[i][j] = 1 + min(
        dp[i-1][j],          # delete ref[i-1]
        dp[i][j-1],          # insert hyp[j-1]
        dp[i-1][j-1]         # substitute hyp[j-1] for ref[i-1]
    )
```

**Worked by hand**:
```
ref = ["hello", "world"]
hyp = ["hello"]

        ""    "hello"
    ""   0      1
"hello" 1      0        ← ref[0]=="hello" equals hyp[0]=="hello", take the diagonal dp[0][0]=0
"world" 2      1        ← ref[1]=="world" differs from hyp[0]=="hello", 1+min(dp[1][1]=0, dp[2][0]=2, dp[1][0]=1) = 1

Result: dp[2][1] = 1, i.e. 1 deletion is needed (delete "world").
WER = 1 / 2 = 0.5 (50% error rate)
```

**Pseudocode**:
```python
def edit_distance(ref_words, hyp_words):
    m, n = len(ref_words), len(hyp_words)
    dp = [[0] * (n+1) for _ in range(m+1)]

    # initialize
    for i in range(m+1):
        dp[i][0] = i
    for j in range(n+1):
        dp[0][j] = j

    # fill the table
    for i in range(1, m+1):
        for j in range(1, n+1):
            if ref_words[i-1] == hyp_words[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])

    return dp[m][n]
```

Now that you understand the algorithm, go and implement `simple_wer`.



In [ ]:
# ─── Exercise 1: LoRA parameter count ─────────────────────────────────
d_model = 512
r = 16
num_matrices = 3  # q_proj, k_proj, v_proj

def lora_param_math(d_model: int, r: int, num_matrices: int):
    """Count LoRA parameters by hand; return (total_lora_params, total_original_params, reduction_pct).

    Replace each raise below, line by line, with the correct assignment (formulas in the "Working it out by hand" table above),
    then uncomment the return line.
    """
    raise NotImplementedError("TODO: compute original_params_per_matrix = d_model * d_model")
    raise NotImplementedError("TODO: compute lora_params_per_matrix = 2 * d_model * r (the A + B matrices)")
    raise NotImplementedError("TODO: compute total_lora_params = num_matrices * lora_params_per_matrix")
    raise NotImplementedError("TODO: compute total_original_params = num_matrices * original_params_per_matrix")
    raise NotImplementedError("TODO: compute reduction_pct = (1 - total_lora_params / total_original_params) * 100")
    # return total_lora_params, total_original_params, reduction_pct

# Check (runs automatically once implemented; skips gracefully when not, so Run All isn't interrupted)
try:
    total_lora_params, total_original_params, reduction_pct = lora_param_math(
        d_model, r, num_matrices
    )
    expected_lora = 3 * 2 * 512 * 16  # = 49152
    expected_reduction = (1 - expected_lora / (3 * 512 * 512)) * 100  # ≈ 93.75%
    assert total_lora_params == expected_lora, f'Wrong parameter count: {total_lora_params}'
    assert abs(reduction_pct - expected_reduction) < 0.01, f'Wrong reduction: {reduction_pct:.2f}%'
    print(f'✅ LoRA parameters: {total_lora_params:,}, reduction {reduction_pct:.1f}%')
except (NotImplementedError, TypeError):
    print('⏳ Exercise 1 (lora_param_math) not done yet — finish the TODO and run again')

# ─── Exercise 2: a simplified WER without jiwer ────────────────────────
def simple_wer(ref: str, hyp: str) -> float:
    """Single-sentence word-level WER (edit distance / number of reference words), without jiwer.
    ref / hyp are plain strings, split on spaces internally.
    hint: use dynamic programming to compute the edit distance between the tokenized ref and hyp.
    """
    # TODO: implement
    raise NotImplementedError('Please implement simple_wer')

# Check (runs once implemented)
try:
    result = simple_wer("the cat sat on the mat", "the cat sat on a mat")
    assert abs(result - 1/6) < 1e-6, f'Wrong WER: {result}'
    print(f'✅ simple_wer passed: WER = {result:.4f}')
except (NotImplementedError, TypeError):
    print('⏳ simple_wer not implemented yet — finish the TODO and run again')

## Parameter experiment: watch the training curves and catch overfitting

**Knobs to turn and what to expect**:

| Parameter | Suggested values | What to expect |
|---|---|---|
| `r` (LoRA rank) | 4 / 8 / 16 | larger r is more expressive but riskier for overfitting; r=8 is usually best |
| `lora_alpha` | `2r` or `4r` | alpha/r is the actual scaling factor; raising it is like raising the learning rate |
| `num_train_epochs` | 3–5 | after epoch 3, eval WER usually stops falling while train loss keeps dropping → overfitting |
| `learning_rate` | 1e-4 / 5e-5 | 1e-4 converges fast; 5e-5 is steadier but needs more epochs |
| `lora_dropout` | 0.0 / 0.05 | dropout=0.05 regularizes small datasets |

**How to monitor**: the `loss` and `eval_wer` in TensorBoard or in `trainer.state.log_history`. When `eval_wer` stops falling for 2 eval steps in a row while `loss` is still dropping, overfitting has begun.


## 4. Adapting to a low-resource language: Welsh in practice

Whisper switches its target language through the `language` argument of `WhisperProcessor`. Welsh is a language Whisper saw comparatively little of in training, and a few things change compared with English:

| What differs | English (LibriSpeech) | Welsh (Common Voice cy / FLEURS cy_gb) |
|---|---|---|
| Token granularity | split on spaces | split on spaces, but initial mutations (*Cymru* → *Gymru*) change the first letter of a word, so word-level matching is harsh |
| Evaluation metric | word error rate WER | WER plus character error rate CER |
| Text normalization | `BasicTextNormalizer` | strip punctuation, keep the circumflex vowels (â, ŵ, ŷ) consistent |
| Dataset | `librispeech_asr` | `mozilla-foundation/common_voice_13_0` `cy` or `google/fleurs` `cy_gb` |
| Processor language | `language="English"` | `language="Welsh"` |

> **Character error rate (CER)**: remove all spaces first, then compute a character-level edit distance — the same formula as WER, but the unit is a single letter, so a one-letter mutation costs one character instead of a whole word.

In [ ]:
# Demo: adapting Whisper fine-tuning to a low-resource language (Welsh)
# Real training needs access to the Common Voice cy or FLEURS cy_gb dataset

# Step 1: switch the Processor language to Welsh (needs transformers)
if HAS_HF:
    processor_target = WhisperProcessor.from_pretrained(
        "openai/whisper-small",
        language="Welsh",   # switch to Welsh
        task="transcribe"
    )
    print("✅ processor_target switched to Welsh")
else:
    print("⏭️ Skipping the Welsh Processor (needs transformers)")

# Step 2: character-level CER (remove spaces, compute the edit distance over characters)
if HAS_JIWER:
    def compute_cer(reference: str, hypothesis: str) -> float:
        """Character error rate: split into a list of characters, then let jiwer compute the WER over them.

        Key point: why ' '.join(list(chars))?
        ─────────────────────────────────────────
        jiwer.wer() takes strings and splits them into words on spaces internally.
        Feed it "bore da" as is and jiwer sees 2 words, not 6 characters — a word-level score, not CER.

        The fix:
        1. reference.replace(' ', '') removes the spaces between words
        2. list(...) splits the string into a list of characters: ["b", "o", "r", "e"]
        3. ' '.join(...) glues them back together with spaces: ["b", "o", "r", "e"] → "b o r e"
        4. now jiwer treats every letter as a "word" and computes the character-level edit distance correctly

        Example:
          ref = "bore" (4 characters, "morning")
          hyp = "boreau" (6 characters, "mornings")
          list(ref) = ["b", "o", "r", "e"]
          list(hyp) = ["b", "o", "r", "e", "a", "u"]
          ' '.join(list(ref)) = "b o r e"  ← now jiwer sees 4 "words"
          ' '.join(list(hyp)) = "b o r e a u"  ← now jiwer sees 6 "words"
          edit distance = 2 (insert "a" and "u"), CER = 2 / 4 = 50%
        """
        ref_chars = list(reference.replace(' ', ''))
        hyp_chars = list(hypothesis.replace(' ', ''))
        return jiwer.wer(' '.join(ref_chars), ' '.join(hyp_chars))

    # Unit test
    ref_target = "bore da i chi"   # "good morning to you": 10 letters
    hyp_target = "bore da i ti"  # last word misheard: 2 character edits (2/10)
    cer_val = compute_cer(ref_target, hyp_target)
    expected_cer = 2 / 10
    assert abs(cer_val - expected_cer) < 1e-6, f'CER computed wrongly: {cer_val:.4f}, expected {expected_cer:.4f}'
    print(f'✅ CER unit test passed: CER = {cer_val:.4f} (2/10 characters wrong)')
else:
    print("⏭️ jiwer not installed, skipping the CER demo (remove spaces, then a word-level edit distance over characters gives the CER)")

# Step 3: swap the dataset (needs real access)
# dataset_target = load_dataset(
#     "mozilla-foundation/common_voice_13_0", "cy",
#     split={"train": "train", "test": "test"},
#     trust_remote_code=True,
# )
# or: dataset_target = load_dataset('google/fleurs', 'cy_gb', split='train')

# Step 4: a compute_metrics callback for Welsh (character-level CER, needs transformers + jiwer)
if HAS_HF and HAS_JIWER:
    def compute_metrics_target(pred):
        pred_ids  = pred.predictions
        label_ids = pred.label_ids
        label_ids[label_ids == -100] = processor_target.tokenizer.pad_token_id
        pred_str  = processor_target.batch_decode(pred_ids,  skip_special_tokens=True)
        label_str = processor_target.batch_decode(label_ids, skip_special_tokens=True)
        cer = sum(compute_cer(r, h) for r, h in zip(label_str, pred_str)) / len(label_str)
        return {'cer': cer}

    print('✅ Welsh adaptation components ready: processor_target / compute_cer / compute_metrics_target')
    print('Replace compute_metrics with compute_metrics_target and set metric_for_best_model="cer" in TrainingArguments to fine-tune for Welsh')
else:
    print('(Once transformers + jiwer are installed in the cloud, you can assemble compute_metrics_target for Welsh fine-tuning)')

In [ ]:
# Start training (needs a GPU; on a CPU it would be extremely slow, flow check only)
if HAS_HF:
    import torch
    if torch.cuda.is_available():
        trainer = Seq2SeqTrainer(
            model=model,
            args=training_args,
            train_dataset=dataset["train"],
            eval_dataset=dataset["test"],
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            tokenizer=processor.tokenizer,    # transformers<4.40: tokenizer=; newer: processing_class=processor
        )
        trainer.train()
        # After training, look at the best checkpoint
        print("Best checkpoint:", trainer.state.best_model_checkpoint)
        print("Best WER:", trainer.state.best_metric)
    else:
        print("⚠️  No GPU detected, skipping training. Run this in a cloud GPU environment (e.g. Colab A100 / RunPod A100).")
else:
    print("⏭️ transformers/torch not installed, skipping the training demo.")
    print("   Cloud GPU setup, resuming from checkpoints and cost control: see docs/current/course/cloud_gpu_plan.md")

# Optional: plot the loss / WER curves
# import pandas as pd, matplotlib.pyplot as plt
# log_df = pd.DataFrame(trainer.state.log_history)
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# log_df.dropna(subset=["loss"]).plot("step", "loss", ax=axes[0], title="Train Loss")
# log_df.dropna(subset=["eval_wer"]).plot("step", "eval_wer", ax=axes[1], title="Eval WER")
# plt.tight_layout(); plt.show()

## 🎯 The payoff later

What you worked out yourself today — **LoRA = freeze W + train only the low-rank B·A** — comes back in **L84 (LoRA from scratch)**. There you no longer call `peft`; instead you **write** the forward and backward passes of `LoRALinear` yourself, turning what you "used" on Whisper today into something you "build".

The same low-rank idea will also show up in:
- **L73 / L74**: whether the fine-tuned model is any good is quantified by a sentence-by-sentence S/D/I diagnosis of WER — LoRA only "changes" the model; evaluation tells you whether it "changed it right".
- **The parameter-efficient fine-tuning (PEFT) family**: QLoRA, Adapters and Prefix-tuning are all variants of "freeze the backbone, train only a small handful of parameters"; today's `d·r + r·k ≪ d·k` sum is the foundation they share.

One line worth remembering: **in the age of large models, being able to change a model usually doesn't come from changing many parameters, but from changing them in the right low-dimensional subspace.**


## Wrap-up

After injecting LoRA with `get_peft_model`, `Seq2SeqTrainer` updates only the low-rank matrices of `q_proj`/`v_proj`, and within 3 epochs can produce transcripts with WER < 5% on LibriSpeech test-clean. Aurora's `aurora.audio.mel.mel_spectrogram` mel preprocessing is fully aligned with `WhisperProcessor`, and can be reused for an offline feature cache to speed up the data pipeline. The next lesson (L73) computes the word error rate (WER) of real transcripts: implement the edit distance, count the rates of each S/D/I error type, and pinpoint the sentences with the highest WER.


## ✏️ Whiteboard challenge: derive how LoRA works by hand (target: 10 minutes)

Cover the screen and derive on paper:

**Question 1**: why does LoRA use two matrices B·A rather than one low-rank matrix C?
(Initialization: B=0, A randomly initialized from N(0,σ²) → initial ΔW=B·A=0, which leaves the pretrained weights untouched; using C directly can't guarantee a zero initialization)

**Question 2**: if d=1024, k=1024, r=4, how many LoRA parameters are there? How many full parameters? What's the saving?
(LoRA: 1024×4+4×1024=8192; full: 1024²=1,048,576; saving=1-8192/1048576≈99.2%)

**Question 3**: why does Whisper fine-tuning usually inject LoRA only into q_proj and v_proj, rather than every weight?
(The attention query/value decide "what to attend to" and are the key to task adaptation; changing the other layers (k_proj, out, FFN) gives diminishing returns, and sticking to q+v keeps the parameter count down — in practice q+v is enough)

**Question 4**: is WER > 1.0 possible? When does it happen?
(Yes: when there are very many inserted words, e.g. ref="good" (1 word), hyp="this is very very good" (5 words) → WER=(0+0+4)/1=4.0)

**Question 5**: does LoRA add latency at inference time? Why?
(In principle no: before inference, merge B·A back into W (W_new=W+B·A); the parameter count is unchanged and there's zero extra latency; the separate form is used during training only so that just the A/B gradients get updated)

Once you've derived them, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell

# Q1: LoRA zero initialization (concept check)
import numpy as np
d, k, r = 512, 512, 8
B = np.zeros((d, r))
A = np.random.randn(r, k) * 0.01
delta_W = B @ A
assert np.allclose(delta_W, 0), "The initial ΔW=B·A should be all zeros (B is zero-initialized)"
print(f"Q1 ✅  initial ΔW=B·A: max|ΔW|={np.abs(delta_W).max():.2e} (all zeros, pretrained weights untouched)")

# Q2: parameter count
d2, k2, r2 = 1024, 1024, 4
lora_params = d2 * r2 + r2 * k2
full_params = d2 * k2
savings_pct = (1 - lora_params / full_params) * 100
assert lora_params == 8192, f"LoRA params={lora_params}, expected 8192"
assert full_params == 1048576, f"full params={full_params}, expected 1048576"
print(f"Q2 ✅  d={d2},k={k2},r={r2}: LoRA={lora_params:,} vs full={full_params:,}, saving {savings_pct:.1f}%")

# Q3: scope of LoRA in Whisper (concept check — via an actual parameter ratio)
d_ws, r_ws, n_layers = 768, 8, 12   # Whisper-small: d_model=768 (see cell 13)
lora_qv = n_layers * 2 * (d_ws * r_ws + r_ws * d_ws)
full_attn_qkvo = n_layers * 4 * d_ws * d_ws
print(f"Q3 ✅  q+v LoRA params={lora_qv:,} ({n_layers} layers): of the full attention params {full_attn_qkvo:,} that is {lora_qv/full_attn_qkvo*100:.1f}%")

# Q4: checking WER > 1.0
try:
    wer_val = simple_wer("good", "this is very very good")
    assert wer_val > 1.0, f"WER={wer_val:.2f}; with 4 inserted words it should be >1.0"
    print(f"Q4 ✅  simple_wer('good', 'this is very very good')={wer_val:.2f}>1.0 (with many insertions WER>100%)")
except (NotImplementedError, NameError):
    # check by hand
    ref_words = ["good"]
    hyp_words = ["this", "is", "very", "very", "good"]
    # edit_distance = 4 insertions, N = 1
    manual_wer = 4.0 / 1
    print(f"Q4 ✅  WER 'good' vs 'this is very very good' ≈{manual_wer:.1f}>1 (check again once simple_wer is implemented)")

# Q5: LoRA inference latency (numerical check of the merge)
W = np.random.randn(d, k) * 0.1
B_trained = np.random.randn(d, r) * 0.01
A_trained = np.random.randn(r, k) * 0.01
W_merged = W + B_trained @ A_trained  # merged for inference
x = np.random.randn(k)
out_lora = W @ x + (B_trained @ (A_trained @ x))  # LoRA kept separate (training)
out_merged = W_merged @ x                          # after merging (inference)
assert np.allclose(out_lora, out_merged, atol=1e-10), "The outputs before and after merging should be identical"
print(f"Q5 ✅  merge check: max|out_lora - out_merged|={np.abs(out_lora-out_merged).max():.2e} (zero extra inference latency)")
print("\n🎉 LoRA whiteboard challenge passed!")

In [ ]:
try:
    # ✏️ Self-review for this lesson
    l72_review = {
        "lora_math_understood":     None,  # Understand LoRA: W_new=W+B·A, B zero-initialized, only A/B trained? True/False
        "param_count_derivation":   None,  # Can count LoRA parameters and the saving by hand for a given d/k/r? True/False
        "simple_wer_impl":          None,  # simple_wer() implemented correctly, all 3 acceptance criteria pass? True/False
        "trainer_pipeline":         None,  # Understand the Seq2SeqTrainer flow: DataCollator→TrainingArgs→train()→evaluate()? True/False
        "whiteboard_passed":        None,  # All 5 whiteboard derivations done? True/False
    }

    unfilled = [k for k, v in l72_review.items() if v is None]
    assert not unfilled, f'Not filled in yet: {unfilled}'
    weak = [k for k, v in l72_review.items() if v is False]
    if weak:
        print(f'⚠️  Needs more work: {weak}')
    else:
        print('✅ L72 all clear! On to L73: WER evaluation')
except (NotImplementedError, TypeError):
    print('⬜ Finish the TODO exercises above first, then run this cell')

---

→ **Next lesson** · [L73 · Evaluating with WER](L73_wer_eval.ipynb)

> Next lesson covers **WER evaluation**: word error rate (insertions / deletions / substitutions), sentence-level analysis compared with jiwer.